# Clinical CVD Model Improvement — Tuned and Leakage-Aware

This notebook builds an improved clinical-only pipeline from `Clinical Dataset.xlsx`.

**Important:** No method can guarantee 95%+ accuracy, precision, recall, F1 or ROC-AUC on unseen data. Do not force a target by leakage, duplicate records across splits, test-set tuning, or fabricated metrics. The aim is reliable generalization.

## What this version changes
- Cleans implausible clinical measurements and engineers age, BMI, pulse pressure and mean arterial pressure.
- Drops ID and the original age-in-days field.
- Creates stratified train / validation / test sets.
- Tunes XGBoost, LightGBM and CatBoost using cross-validation on the training split only.
- Uses validation data to select a decision threshold for F1.
- Uses the test set once, after model/threshold selection.
- Trains a regularized TensorFlow DNN with early stopping as an additional comparison.
- Saves improved artifacts in a separate folder so existing fusion artifacts are not silently overwritten.

Run from the project root or update `DATA_PATH` below.

In [3]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, classification_report, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path

# Automatically detect the project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "Trained Models":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = PROJECT_ROOT / "DataSets" / "Clinical Dataset.xlsx"

OUT_DIR = PROJECT_ROOT / "Trained Models" / "clinical_improved"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)
print("Output folder:", OUT_DIR)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check that Clinical Dataset.xlsx exists inside DataSets."
    )
# If running this notebook from a subfolder, set PROJECT_ROOT explicitly, e.g.:
# PROJECT_ROOT = Path(r"C:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14")

DATA_PATH = PROJECT_ROOT / "DataSets" / "Clinical Dataset.xlsx"
OUT_DIR = PROJECT_ROOT / "Trained Models" / "clinical_improved"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset exists:", DATA_PATH.exists(), DATA_PATH)
print("Output folder:", OUT_DIR)
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Set PROJECT_ROOT or DATA_PATH to your project location."
    )

Project root: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14
Dataset exists: True
Dataset path: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\DataSets\Clinical Dataset.xlsx
Output folder: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\Trained Models\clinical_improved
Project root: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14
Dataset exists: True c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\DataSets\Clinical Dataset.xlsx
Output folder: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\Trained Models\clinical_improved


## 1. Load and inspect data

In [4]:
df = pd.read_excel(DATA_PATH)
print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
display(df["cardio"].value_counts(dropna=False).rename("count").to_frame())
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

Raw shape: (70000, 13)
Columns: ['id', 'age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


,count
cardio,
0,35021
1,34979


Missing values: 0
Duplicate rows: 0


## 2. Clean records and engineer features

In [5]:
required = {
    "gender", "height", "weight", "ap_hi", "ap_lo",
    "cholesterol", "gluc", "smoke", "alco", "active", "cardio"
}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

df = df.drop_duplicates().copy()

# Remove physiologically implausible measurements; report how many rows are removed.
valid = (
    df["ap_hi"].gt(0) &
    df["ap_lo"].gt(0) &
    df["ap_hi"].le(300) &
    df["ap_lo"].le(200) &
    df["ap_hi"].gt(df["ap_lo"]) &
    df["height"].between(100, 220) &
    df["weight"].between(30, 200)
)
before = len(df)
df = df.loc[valid].copy()
print(f"Rows removed by measurement filters: {before - len(df):,} ({(before-len(df))/before:.2%})")
print("Rows remaining:", len(df))

# Age is commonly recorded in days in this dataset.
if "age" in df.columns:
    df["age_years"] = df["age"] / 365.25
    df = df.drop(columns=["age"])
else:
    raise ValueError("Expected 'age' column to derive age_years.")

df["BMI"] = df["weight"] / ((df["height"] / 100.0) ** 2)
df["pulse_pressure"] = df["ap_hi"] - df["ap_lo"]
df["MAP"] = (df["ap_hi"] + 2 * df["ap_lo"]) / 3.0

# ID is an identifier, not a predictive clinical feature.
df = df.drop(columns=["id"], errors="ignore")

# Match the existing notebook's gender mapping when raw values are 1/2.
if set(pd.Series(df["gender"].dropna().unique()).tolist()).issubset({1, 2}):
    df["gender"] = df["gender"].map({1: 0, 2: 1})

# One-hot encode ordered categories without imposing a false numeric distance.
df = pd.get_dummies(
    df, columns=["cholesterol", "gluc"], drop_first=True, dtype=int
)

df = df.dropna(subset=["cardio"]).copy()
df["cardio"] = df["cardio"].astype(int)
if not set(df["cardio"].unique()).issubset({0, 1}):
    raise ValueError("Target cardio must be binary 0/1.")

X = df.drop(columns=["cardio"])
y = df["cardio"]
# Handle any remaining missing feature values using training-set medians later.
print("Model matrix:", X.shape)
print("Target distribution (%):")
display((y.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("percent"))
display(X.describe().T)

Rows removed by measurement filters: 1,325 (1.89%)
Rows remaining: 68675
Model matrix: (68675, 16)
Target distribution (%):


,percent
cardio,
0,50.53
1,49.47


,count,mean,std,min,25%,50%,75%,max
gender,68675.0,0.348584,0.476525,0.000000,0.000000,0.000000,1.000000,1.000000
height,68675.0,164.394539,7.976723,100.000000,159.000000,165.000000,170.000000,207.000000
weight,68675.0,74.117975,14.307375,30.000000,65.000000,72.000000,82.000000,200.000000
ap_hi,68675.0,126.671846,16.708720,16.000000,120.000000,120.000000,140.000000,240.000000
ap_lo,68675.0,81.273855,9.568023,1.000000,80.000000,80.000000,90.000000,182.000000
smoke,68675.0,0.087907,0.283161,0.000000,0.000000,0.000000,0.000000,1.000000
alco,68675.0,0.053338,0.224709,0.000000,0.000000,0.000000,0.000000,1.000000
active,68675.0,0.803393,0.397436,0.000000,1.000000,1.000000,1.000000,1.000000
age_years,68675.0,53.290610,6.756940,29.563313,48.343600,53.938398,58.381930,64.922656
BMI,68675.0,27.472735,5.351504,10.726644,23.875115,26.346494,30.119376,152.551775


## 3. Create stratified train / validation / test splits

In [6]:
# 70% train, 15% validation, 15% test. Test remains untouched until final evaluation.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=(0.15 / 0.85),
    random_state=SEED, stratify=y_trainval
)

# Fit imputation values from training data only, then apply them to val/test.
train_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_medians)
X_val = X_val.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# Align schemas and enforce numeric inputs.
feature_names = X_train.columns.tolist()
X_val = X_val.reindex(columns=feature_names, fill_value=0)
X_test = X_test.reindex(columns=feature_names, fill_value=0)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)
for name, yy in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(name, "class proportions:", yy.value_counts(normalize=True).sort_index().round(3).to_dict())

Train: (48071, 16) Validation: (10302, 16) Test: (10302, 16)
train class proportions: {0: 0.505, 1: 0.495}
validation class proportions: {0: 0.505, 1: 0.495}
test class proportions: {0: 0.505, 1: 0.495}


## 4. Tune tree models using training-only cross-validation

Search scoring is ROC-AUC to select probability ranking quality. We will choose a practical operating threshold separately using validation data. This search can take several minutes depending on your CPU.

In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

search_spaces = {}
try:
    from xgboost import XGBClassifier
    search_spaces["XGBoost"] = (
        XGBClassifier(
            random_state=SEED, eval_metric="logloss", tree_method="hist",
            n_jobs=1
        ),
        {
            "n_estimators": [200, 350, 500, 700],
            "max_depth": [2, 3, 4, 5, 6],
            "learning_rate": [0.01, 0.02, 0.04, 0.06, 0.08],
            "subsample": [0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
            "min_child_weight": [1, 3, 5, 8],
            "reg_lambda": [1, 2, 5, 10],
            "gamma": [0, 0.1, 0.3]
        }
    )
except ImportError:
    print("XGBoost not installed; skipping. Install with: pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    search_spaces["LightGBM"] = (
        LGBMClassifier(random_state=SEED, verbosity=-1, n_jobs=1),
        {
            "n_estimators": [200, 350, 500, 700],
            "learning_rate": [0.01, 0.02, 0.04, 0.06, 0.08],
            "num_leaves": [7, 15, 23, 31, 45],
            "max_depth": [-1, 3, 4, 5, 7],
            "min_child_samples": [15, 25, 40, 60],
            "subsample": [0.7, 0.8, 0.9, 1.0],
            "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
            "reg_lambda": [0, 1, 3, 7]
        }
    )
except ImportError:
    print("LightGBM not installed; skipping. Install with: pip install lightgbm")

try:
    from catboost import CatBoostClassifier
    search_spaces["CatBoost"] = (
        CatBoostClassifier(random_seed=SEED, verbose=False, thread_count=1, allow_writing_files=False),
        {
            "iterations": [250, 400, 600, 800],
            "depth": [4, 5, 6, 7, 8],
            "learning_rate": [0.01, 0.02, 0.04, 0.06, 0.08],
            "l2_leaf_reg": [2, 4, 6, 8, 10],
            "random_strength": [0.2, 0.5, 1, 2]
        }
    )
except ImportError:
    print("CatBoost not installed; skipping. Install with: pip install catboost")

if not search_spaces:
    raise ImportError("Install at least one of xgboost, lightgbm, catboost to run tuning.")

tuned_models = {}
cv_rows = []
for name, (estimator, params) in search_spaces.items():
    print(f"\nTuning {name}...")
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=params,
        n_iter=25,
        scoring="roc_auc",
        cv=cv,
        random_state=SEED,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=True
    )
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    cv_rows.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Best params": search.best_params_
    })
    print("Best CV ROC-AUC:", round(search.best_score_, 4))
    print("Best params:", search.best_params_)

cv_results = pd.DataFrame(cv_rows).sort_values("Best CV ROC-AUC", ascending=False)
display(cv_results)


Tuning XGBoost...
Fitting 5 folds for each of 25 candidates, totalling 125 fits


c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
3 fits failed out of a total of 125.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\.venv\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\.venv\Lib\

Best CV ROC-AUC: 0.8033
Best params: {'subsample': 0.7, 'reg_lambda': 2, 'n_estimators': 700, 'min_child_weight': 1, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0.1, 'colsample_bytree': 1.0}

Tuning LightGBM...
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best CV ROC-AUC: 0.803
Best params: {'subsample': 0.7, 'reg_lambda': 7, 'num_leaves': 31, 'n_estimators': 500, 'min_child_samples': 40, 'max_depth': -1, 'learning_rate': 0.01, 'colsample_bytree': 0.7}

Tuning CatBoost...
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best CV ROC-AUC: 0.8032
Best params: {'random_strength': 0.2, 'learning_rate': 0.01, 'l2_leaf_reg': 8, 'iterations': 600, 'depth': 8}


,Model,Best CV ROC-AUC,Best params
0,XGBoost,0.803299,"{'subsample': 0.7, 'reg_lambda': 2, 'n_estimat..."
2,CatBoost,0.803168,"{'random_strength': 0.2, 'learning_rate': 0.01..."
1,LightGBM,0.802965,"{'subsample': 0.7, 'reg_lambda': 7, 'num_leave..."


## 5. Choose an F1 threshold on validation data only

In [8]:
def metrics_at_threshold(y_true, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "Balanced accuracy": balanced_accuracy_score(y_true, predictions),
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities),
    }

validation_rows = []
thresholds_by_model = {}
for name, model in tuned_models.items():
    p_val = model.predict_proba(X_val)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_val, p_val)
    # precision/recall have one more element than thresholds. Use only threshold-aligned entries.
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    if len(thresholds) == 0:
        best_threshold = 0.5
    else:
        best_threshold = float(thresholds[int(np.nanargmax(f1_values))])
    thresholds_by_model[name] = best_threshold
    row = {"Model": name, "Threshold": best_threshold}
    row.update(metrics_at_threshold(y_val, p_val, best_threshold))
    validation_rows.append(row)

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["ROC-AUC", "F1"], ascending=False
).reset_index(drop=True)
display(validation_results.round(4))

# Select the model based on validation ROC-AUC, with F1 as tie-breaker.
best_name = validation_results.sort_values(
    ["ROC-AUC", "F1"], ascending=False
).iloc[0]["Model"]
best_model = tuned_models[best_name]
best_threshold = thresholds_by_model[best_name]
print("Selected using validation metrics:", best_name)
print("Selected validation threshold:", round(best_threshold, 4))

,Model,Threshold,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC
0,CatBoost,0.3608,0.7149,0.6746,0.8185,0.7397,0.7160,0.7991,0.7801
1,XGBoost,0.3394,0.7056,0.6585,0.8413,0.7387,0.7070,0.7990,0.7827
2,LightGBM,0.3681,0.7169,0.6809,0.8054,0.7379,0.7179,0.7984,0.7794


Selected using validation metrics: CatBoost
Selected validation threshold: 0.3608


## 6. Train and evaluate a regularized DNN as a separate comparison

This is an additional baseline, not guaranteed to outperform boosted trees. Early stopping monitors validation loss; test data are not used for training decisions.

In [9]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.keras.utils.set_random_seed(SEED)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

dnn = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.35),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.25),
    Dense(32, activation="relu"),
    Dropout(0.15),
    Dense(1, activation="sigmoid")
], name="Clinical_DNN_Improved")

dnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc")
    ]
)
callbacks = [
    EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
class_ids = np.unique(y_train)
class_weights_arr = compute_class_weight(class_weight="balanced", classes=class_ids, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(class_ids, class_weights_arr)}

history = dnn.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=128,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

dnn_val_prob = dnn.predict(X_val_scaled, verbose=0).ravel()
p, r, t = precision_recall_curve(y_val, dnn_val_prob)
f1s = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
dnn_threshold = float(t[int(np.nanargmax(f1s))]) if len(t) else 0.5
print("DNN validation threshold:", round(dnn_threshold, 4))
print("DNN validation metrics:", metrics_at_threshold(y_val, dnn_val_prob, dnn_threshold))

Epoch 1/100
376/376 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7013 - auc: 0.7595 - loss: 0.5971 - val_accuracy: 0.7309 - val_auc: 0.7944 - val_loss: 0.5531 - learning_rate: 0.0010
Epoch 2/100
376/376 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7252 - auc: 0.7862 - loss: 0.5611 - val_accuracy: 0.7314 - val_auc: 0.7966 - val_loss: 0.5476 - learning_rate: 0.0010
Epoch 3/100
376/376 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7280 - auc: 0.7915 - loss: 0.5546 - val_accuracy: 0.7317 - val_auc: 0.7970 - val_loss: 0.5471 - learning_rate: 0.0010
Epoch 4/100
376/376 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7323 - auc: 0.7941 - loss: 0.5510 - val_accuracy: 0.7317 - val_auc: 0.7970 - val_loss: 0.5472 - learning_rate: 0.0010
Epoch 5/100
376/376 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7317 - auc: 0.7946 - loss: 0.5503 - val_accuracy: 0.7329 - val_auc: 0.7969 - val_loss: 0.5466 - learning_rate: 0.0010
Epoch 6/100
376/376 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7331 - a

## 7. Final test evaluation — run once after model selection

The test metrics below are estimates for this particular split. Avoid making more model/threshold changes after looking at them; if you do, use a new untouched test cohort or nested evaluation.

In [10]:
# Evaluate each tuned tree model on the held-out test set using its validation-selected threshold.
# This comparison is for reporting; the selected model was chosen only from validation metrics.
final_rows = []
for name, model in tuned_models.items():
    p_test = model.predict_proba(X_test)[:, 1]
    threshold = thresholds_by_model[name]
    row = {"Model": name, "Threshold": threshold}
    row.update(metrics_at_threshold(y_test, p_test, threshold))
    final_rows.append(row)

# DNN comparison uses its threshold chosen on validation data.
dnn_test_prob = dnn.predict(X_test_scaled, verbose=0).ravel()
dnn_row = {"Model": "DNN", "Threshold": dnn_threshold}
dnn_row.update(metrics_at_threshold(y_test, dnn_test_prob, dnn_threshold))
final_rows.append(dnn_row)

final_results = pd.DataFrame(final_rows).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
display(final_results.round(4))

# Only this selected model gets the headline classification report and confusion matrix.
selected_test_prob = best_model.predict_proba(X_test)[:, 1]
selected_test_pred = (selected_test_prob >= best_threshold).astype(int)
print(f"Selected model: {best_name}")
print("Test classification report:")
print(classification_report(y_test, selected_test_pred, target_names=["No CVD", "CVD"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, selected_test_pred))
print("Selected-model test metrics:")
print(pd.Series(metrics_at_threshold(y_test, selected_test_prob, best_threshold)).round(4))

,Model,Threshold,Accuracy,Precision,Recall,F1,Balanced accuracy,ROC-AUC,PR-AUC
0,LightGBM,0.3681,0.7108,0.6744,0.8034,0.7333,0.7118,0.7968,0.7790
1,XGBoost,0.3394,0.6982,0.6515,0.8385,0.7333,0.6997,0.7967,0.7794
2,CatBoost,0.3608,0.7067,0.6672,0.8122,0.7326,0.7078,0.7963,0.7798
3,DNN,0.3504,0.7041,0.6599,0.8297,0.7351,0.7054,0.7951,0.7769


Selected model: CatBoost
Test classification report:
              precision    recall  f1-score   support

      No CVD       0.77      0.60      0.68      5205
         CVD       0.67      0.81      0.73      5097

    accuracy                           0.71     10302
   macro avg       0.72      0.71      0.70     10302
weighted avg       0.72      0.71      0.70     10302

Confusion matrix:
[[3140 2065]
 [ 957 4140]]
Selected-model test metrics:
Accuracy             0.7067
Precision            0.6672
Recall               0.8122
F1                   0.7326
Balanced accuracy    0.7078
ROC-AUC              0.7963
PR-AUC               0.7798
dtype: float64


## 8. Save the improved clinical model and reproducibility metadata

In [11]:
# Save in a separate directory: do not overwrite existing fusion artifacts automatically.
model_path = OUT_DIR / f"{best_name.lower()}_tuned.joblib"
joblib.dump(best_model, model_path)
joblib.dump(feature_names, OUT_DIR / "clinical_improved_features.pkl")
joblib.dump(train_medians, OUT_DIR / "clinical_improved_medians.pkl")
joblib.dump(scaler, OUT_DIR / "clinical_improved_scaler.pkl")
dnn.save(OUT_DIR / "clinical_dnn_improved.keras")

metadata = {
    "selected_model": best_name,
    "selected_threshold": float(best_threshold),
    "feature_names": feature_names,
    "target": "cardio",
    "target_mapping": {"0": "no CVD", "1": "CVD"},
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15, "random_state": SEED},
    "selection": "Highest validation ROC-AUC, F1 as tie-breaker; threshold chosen on validation for F1",
    "test_metrics": {k: float(v) for k, v in metrics_at_threshold(y_test, selected_test_prob, best_threshold).items()},
    "scientific_note": "Performance is dataset/split dependent; no 95%+ target is guaranteed. Validate externally before clinical claims."
}
with open(OUT_DIR / "clinical_improved_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved selected model:", model_path)
print("Saved metadata:", OUT_DIR / "clinical_improved_metadata.json")
print("Saved DNN:", OUT_DIR / "clinical_dnn_improved.keras")
print("Note: these artifacts use the improved notebook's feature schema and are separate from existing fusion artifacts.")

Saved selected model: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\Trained Models\clinical_improved\catboost_tuned.joblib
Saved metadata: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\Trained Models\clinical_improved\clinical_improved_metadata.json
Saved DNN: c:\Users\deepa\Desktop\CVD-XAI\SKIT-AI-2023-2027-14\Trained Models\clinical_improved\clinical_dnn_improved.keras
Note: these artifacts use the improved notebook's feature schema and are separate from existing fusion artifacts.


## 9. Interpretation and next steps

- If ROC-AUC is good but F1/precision/recall trade off, choose the threshold based on the project's stated clinical objective; F1 is not automatically the medically correct objective.
- If the results are far below 95%, do not force them upward. Investigate label noise, data source limitations, external validation and whether the task definition supports the claimed performance.
- The current clinical dataset and the Ningbo ECG dataset do not share patient IDs. These clinical-only results do not validate multimodal fusion.
- For a fair model comparison, report class-wise metrics, PR-AUC, ROC-AUC, calibration, and confidence intervals where practical.
- Before deploying this model, test inference on unseen data and ensure the same preprocessing and feature order are used.